<a href="https://colab.research.google.com/github/mdmostafizurrahman6351/Deep_Learning_Project/blob/main/cat_vs_dog_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# packge import
#!pip install tensorflow
import tensorflow as tf
import os, pathlib, random
from PIL import Image
import numpy as np
from tensorflow.keras import layers

In [ ]:
# data download
!wget https://download.microsoft.com/download/3/e/1/3e1c3f21-ecdb-4869-8368-6deba77b919f/kagglecatsanddogs_5340.zip

In [ ]:
# data unzip
!unzip -q kagglecatsanddogs_5340.zip -d dat/

print(os.listdir('dat/'))

In [ ]:
# data cleaning
def clean_data(path):
  removed = 0
  for folder in ['Cat', 'Dog']:
    folder_path = os.path.join(path, folder)

    for file in os.listdir(folder_path):
      file_path = os.path.join(folder_path, file)

      try:
        img = Image.open(file_path)
        img.verify()
      except:
        os.remove(file_path)
        removed += 1
  print(f'Removed {removed} image')

clean_data('/content/dat/PetImages')

In [ ]:
# data load & preprocessing
data_dir = pathlib.Path('/content/dat/PetImages')

batch_size = 32
img_size = (124,124)

train_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset='training',
  seed=123,
  image_size=img_size,
  batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset='validation',
  seed=123,
  image_size=img_size,
  batch_size=batch_size
)

print(train_ds.class_names)

In [ ]:

# data augmantation layee
data_augmentatolion = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])


# data perfornance, normalization, augmantation

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.shuffle(1000)


train_ds = train_ds.map(lambda x, y: (data_augmentatolion(x)/255.0, y))

val_ds = val_ds.map(lambda x, y: (x/255, y))

train_ds = train_ds.apply(tf.data.experimental.ignore_errors())
val_ds = val_ds.apply(tf.data.experimental.ignore_errors())

train_ds = train_ds.cache('train_cache')
val_ds = val_ds.cache('val_cache')

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)


print('complete , data')

In [ ]:
# model create
model = tf.keras.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(124,124,3)),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(256, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

print('model compile')

In [ ]:
# model compile
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
print('model compile')

In [ ]:
# model fit
num_epochs = 20
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=num_epochs
)

In [ ]:
# model evulate
loss, accuracy = model.evaluate(val_ds)
print(f'loss: {loss}, accuracy: {accuracy}')

In [ ]:
# model save in drive
model.save('/content/drive/MyDrive/Colab Notebooks/Cat_vs_Dog-model.h5')
print('model save')

In [ ]:
model.summary()

save model load and prediction system create

In [ ]:
import tensorflow as tf
import numpy as np
from PIL import Image


# model load
model = tf.keras.models.load_model('/content/drive/MyDrive/Colab Notebooks/Cat_vs_Dog-model.h5')



In [ ]:
import requests
import matplotlib.pyplot as plt
from IPython.display import Image as DisplayImage

# prediction system create

def predict_img(img_path):
  if img_path.startswith('http:') or img_path.startswith('https:'):
    # Display image from URL
    display(DisplayImage(url=img_path))
    img = Image.open(requests.get(img_path, stream=True).raw)
  else:
    # Display local image
    plt.imshow(Image.open(img_path))
    plt.axis('off')
    plt.show()
    img = Image.open(img_path)

  img = img.resize((124,124))
  img = np.array(img)

  img = np.expand_dims(img, axis=0)
  img = img/255.0

  prediction = model.predict(img)

  if prediction[0][0] > 0.5:
    print('Dog')
  else:
    print('Cat')

  print(f'Prediction Acccuracy: {prediction[0][0]}')


img = input('enter your image path: ')

predict_img(img)